In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, GridSearchCV, cross_val_predict
)
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    mean_squared_error, accuracy_score,
    precision_score, recall_score, f1_score
)

from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

In [2]:
def compute_metrics(y_true, y_pred, label=""):
    mse  = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    rec  = recall_score(y_true, y_pred, average="weighted", zero_division=0)
    f1   = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    return {
        "Set": label,
        "MSE": round(mse, 4),
        "RMSE": round(rmse, 4),
        "Accuracy": round(acc, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1": round(f1, 4)
    }

In [3]:
def remove_outliers_iqr(df, features):
    for feature in features:
        Q1 = df[feature].quantile(0.25)
        Q3 = df[feature].quantile(0.75)
        IQR = Q3 - Q1
        df = df[(df[feature] >= Q1 - 1.5 * IQR) & (df[feature] <= Q3 + 1.5 * IQR)]
    return df

In [4]:
from sklearn.datasets import fetch_california_housing
california_housing = fetch_california_housing(as_frame=True)
df = california_housing.frame

df_o = remove_outliers_iqr(df, ['MedInc', 'HouseAge','AveRooms','AveBedrms','Population','AveOccup','MedHouseVal'])
df_o = df_o.rename(columns={'MedHouseVal': 'target'})
df_final = df_o.iloc[:,[1,2,3,4,5,6,7,0,8]]
df_final['target'] = df_final.apply(lambda row: 0 if (row['target']<0.5)else 1, axis=1)

X_train, X_test = train_test_split(df_final, test_size=0.2, random_state=42)

scaler = MinMaxScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=df_final.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=df_final.columns)

X_train = X_train_scaled.iloc[:, :-1]
X_test = X_test_scaled.iloc[:, :-1]
y_train = X_train_scaled.iloc[:, -1]
y_test = X_test_scaled.iloc[:, -1]

In [6]:
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

models = {
    "Naive Bayes": {
        "pipeline": Pipeline([
            ("scaler", MinMaxScaler()),
            ("clf", GaussianNB())
        ]),
        "params": {
            "clf__var_smoothing": np.logspace(-12, -6, 7)
        }
    },

    "Logistic Regression": {
        "pipeline": Pipeline([
            ("scaler", MinMaxScaler()),
            ("clf", LogisticRegression(max_iter=1000, random_state=42))
        ]),
        "params": {
            "clf__C": [0.001, 0.01, 0.1, 1, 10, 100],
            "clf__solver": ["lbfgs", "liblinear"],
            "clf__penalty": ["l2"]
        }
    },

    "Random Forest": {
        "pipeline": Pipeline([
            ("scaler", MinMaxScaler()),
            ("clf", RandomForestClassifier(random_state=42))
        ]),
        "params": {
            "clf__n_estimators": [50, 100, 200],
            "clf__max_depth": [None, 5, 10],
            "clf__min_samples_split": [2, 5],
            "clf__min_samples_leaf": [1, 2]
        }
    },

    "Neural Network": {
        "pipeline": Pipeline([
            ("scaler", MinMaxScaler()),
            ("clf", MLPClassifier(max_iter=500, random_state=42))
        ]),
        "params": {
            "clf__hidden_layer_sizes": [(64,), (128,), (64, 32), (128, 64)],
            "clf__activation": ["relu", "tanh"],
            "clf__alpha": [0.0001, 0.001, 0.01],
            "clf__learning_rate_init": [0.001, 0.01]
        }
    }
}

In [7]:
all_results = []

for name, config in models.items():
    print(f"\n{'='*60}")
    print(f"  Model: {name}")
    print(f"{'='*60}")

    gs = GridSearchCV(
        estimator=config["pipeline"],
        param_grid=config["params"],
        cv=cv,
        scoring="f1_weighted",
        n_jobs=-1,
        verbose=0
    )
    gs.fit(X_train, y_train)

    best_model = gs.best_estimator_
    print(f"  The best params : {gs.best_params_}")
    print(f"  CV F1 (val. set)   : {gs.best_score_:.4f}")

    y_train_pred = cross_val_predict(best_model, X_train, y_train, cv=cv)
    train_metrics = compute_metrics(y_train, y_train_pred, label="Train (CV)")

    y_test_pred = best_model.predict(X_test)
    test_metrics = compute_metrics(y_test, y_test_pred, label="Test")

    for m in [train_metrics, test_metrics]:
        print(f"\n  [{m['Set']}]")
        for k, v in m.items():
            if k != "Set":
                print(f"    {k:<12}: {v}")

    all_results.append({"Model": name, **train_metrics})
    all_results.append({"Model": name, **test_metrics})


  Model: Naive Bayes
  The best params : {'clf__var_smoothing': 1e-12}
  CV F1 (val. set)   : 0.9884

  [Train (CV)]
    MSE         : 0.0085
    RMSE        : 0.092
    Accuracy    : 0.9915
    Precision   : 0.9863
    Recall      : 0.9915
    F1          : 0.9884

  [Test]
    MSE         : 0.0102
    RMSE        : 0.1008
    Accuracy    : 0.9898
    Precision   : 0.984
    Recall      : 0.9898
    F1          : 0.9856

  Model: Logistic Regression
  The best params : {'clf__C': 100, 'clf__penalty': 'l2', 'clf__solver': 'liblinear'}
  CV F1 (val. set)   : 0.9885

  [Train (CV)]
    MSE         : 0.0077
    RMSE        : 0.0878
    Accuracy    : 0.9923
    Precision   : 0.9924
    Recall      : 0.9923
    F1          : 0.9885

  [Test]
    MSE         : 0.0099
    RMSE        : 0.0993
    Accuracy    : 0.9901
    Precision   : 0.9804
    Recall      : 0.9901
    F1          : 0.9852

  Model: Random Forest
  The best params : {'clf__max_depth': None, 'clf__min_samples_leaf': 1, 'clf_

In [8]:
print(f"\n\n{'='*60}")
print("Metrics")
print(f"{'='*60}")
df_results = pd.DataFrame(all_results)
df_results = df_results.set_index(["Model", "Set"])
print(df_results.to_string())



Metrics
                                   MSE    RMSE  Accuracy  Precision  Recall      F1
Model               Set                                                            
Naive Bayes         Train (CV)  0.0085  0.0920    0.9915     0.9863  0.9915  0.9884
                    Test        0.0102  0.1008    0.9898     0.9840  0.9898  0.9856
Logistic Regression Train (CV)  0.0077  0.0878    0.9923     0.9924  0.9923  0.9885
                    Test        0.0099  0.0993    0.9901     0.9804  0.9901  0.9852
Random Forest       Train (CV)  0.0071  0.0842    0.9929     0.9911  0.9929  0.9908
                    Test        0.0089  0.0945    0.9911     0.9895  0.9911  0.9878
Neural Network      Train (CV)  0.0069  0.0828    0.9931     0.9915  0.9931  0.9917
                    Test        0.0092  0.0961    0.9908     0.9881  0.9908  0.9880
